## llama.cpp GGUF backend — Colab T4 1x

Three-cell MVP: setup, download, launch + tunnel. Configuration stays in normal code cells; widgets are used only for output.

In [ ]:
# CELL 1 — clone/update repo, install package, diagnostics, dependencies, llama.cpp prebuilt
import os, sys, subprocess
from pathlib import Path

ROOT = "/content"
REPO_URL = "https://github.com/N3iKos/llama-cpp-notebook"
REPO_BRANCH = "main"
REPO_DIR = None
for candidate in [Path.cwd(), Path.cwd().parent, Path(ROOT) / "llama-cpp-notebook"]:
    if (candidate / "gguf_backend" / "workflow.py").exists():
        REPO_DIR = candidate.resolve()
        break

if REPO_DIR is None:
    REPO_DIR = Path(ROOT) / "llama-cpp-notebook"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
sys.path.insert(0, str(REPO_DIR))

from gguf_backend.workflow import setup_runtime

setup_info = setup_runtime(
    root_dir=ROOT,
    cuda_preference="12.8",
    force_llama=False,
    include_diagnostics=True,
    install_pyngrok=True,
)


In [ ]:
# CELL 2 — download model and optional mmproj
import os
from gguf_backend.workflow import download_assets

MODEL_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/Qwen2.5-VL-3B-Instruct-Q4_K_M.gguf"
MMPROJ_URL = "https://huggingface.co/ggml-org/Qwen2.5-VL-3B-Instruct-GGUF/resolve/main/mmproj-Qwen2.5-VL-3B-Instruct-Q8_0.gguf"
MODEL_DIR = f"{ROOT}/models/current"
HF_TOKEN = os.environ.get("HF_TOKEN", "")

model_config = download_assets(
    model_url=MODEL_URL,
    mmproj_url=MMPROJ_URL,
    model_dir=MODEL_DIR,
    hf_token=HF_TOKEN,
    connections=16,
    downloader="auto",
)


In [ ]:
# CELL 3  config, start server, warmup, ngrok/cloudflare tunnel + LIVE MONITOR
from gguf_backend.workflow import launch_backend_live, get_kaggle_secret

# -------------------------
# General config
# -------------------------
GENERAL_CONFIG = {
    "host": "0.0.0.0",              # listen on all interfaces
    "port": 8080,                   # llama-server port
    "alias": "local-vl",            # model name used by clients
    "ctx_size": 8192,               # context window
    "gpu_layers": 999,              # GPU offload layers; 999 = offload as much as possible
    "cuda_visible_devices": "0,1",  # Kaggle T4 x2
    "split_mode": "row",            # multi-GPU split mode
    "fallback_split_mode": "layer", # retry with this split mode if startup fails; "" disables fallback
    "tensor_split": "1,1",          # GPU split ratio for two GPUs
}

# -------------------------
# Advanced config
# Empty string "" means: do not pass the flag; use llama.cpp default.
# For on/off flags, use "on" or "off".
# -------------------------
ADVANCED_CONFIG = {
    # Performance / memory
    "main_gpu": "",          # main GPU index
    "threads": "",           # CPU generation threads
    "threads_batch": "",     # CPU prompt/batch threads
    "threads_http": "",      # HTTP worker threads
    "parallel": "",          # parallel slots
    "batch_size": "",        # prompt processing batch size
    "ubatch_size": "",       # physical micro-batch size
    "flash_attn": "",        # "on" or "off"
    "cache_type_k": "",      # e.g. "f16", "q8_0", "q4_0"
    "cache_type_v": "",      # e.g. "f16", "q8_0", "q4_0"
    "kv_offload": "",        # "on" or "off"
    "cont_batching": "",     # continuous batching: "on" or "off"
    "cache_prompt": "",      # prompt cache: "on" or "off"
    "cache_reuse": "",       # KV cache reuse chunk size
    "mmap": "",              # memory-map model: "on" or "off"
    "mlock": "",             # lock model in RAM: "on" or "off"
    "no_perf": "",           # disable perf timings if "on"
    "log_verbosity": "",     # llama.cpp log verbosity

    # Multimodal / template / reasoning (manual overrides)
    "mmproj_offload": "",            # offload mmproj if supported: "on" or "off"
    "image_min_tokens": "",          # minimum image tokens
    "image_max_tokens": "",          # maximum image tokens
    "chat_template_kwargs": "",      # JSON string; overrides thinking mapper if set
    "chat_template": "",             # built-in/custom chat template name
    "chat_template_file": "",        # path to custom jinja template file
    "jinja": "",                     # jinja template engine: "on" or "off"
    "reasoning": "",                 # manual override: "on", "off", or "auto"
    "reasoning_format": "",          # manual override: e.g. "none", "deepseek"
    "reasoning_budget": "",          # manual override: -1 unlimited, 0 off, N tokens
    "reasoning_budget_message": "",  # message injected when budget is exhausted

    # API / server features
    "timeout": "",           # server read/write timeout seconds
    "api_key": "",           # require API key if set
    "api_key_file": "",      # file containing API keys
    "api_prefix": "",        # custom API prefix
    "ui": "",                # llama.cpp web UI: "on" or "off"
    "metrics": "",           # prometheus metrics endpoint: "on"
    "slots": "",             # slots endpoint: "on" or "off"
    "props": "",             # /props endpoint: "on"
    "embedding": "",         # embeddings-only mode: "on"
    "reranking": "",         # reranking endpoint: "on"
    "slot_save_path": "",    # path for slot KV cache files
    "media_path": "",        # path for local media files

    # Raw llama.cpp args for flags not wrapped yet
    "extra_server_args": [],
}

# -------------------------
# Thinking/reasoning toggle (model-aware)
# Preferred interface for reasoning config. The mapper auto-detects
# model family and translates to the correct llama-server flags.
# Manual overrides in ADVANCED_CONFIG always win if explicitly set.
# -------------------------
THINKING_CONFIG = {
    "family": "auto",      # auto, gemma4, qwen3, deepseek-v31, glm, hermes4, gpt-oss, none
    "mode": "off",         # on/off/auto (for gpt-oss: low/medium/high)
    "budget": "0",         # -1 unlimited, 0 minimal/off, N token budget, "" = default
    "format": "",          # none, deepseek, deepseek-legacy, "" = default
    "soft_prompt": "",     # hint only; shown in dashboard, not injected into prompts
}

# -------------------------
# Tunnel config
# -------------------------
TUNNEL_MODE = "both"          # "both", "ngrok", "cloudflare", or "none"
NGROK_AUTHTOKEN = ""          # optional
NGROK_AUTHTOKEN = NGROK_AUTHTOKEN or get_kaggle_secret("NGROK_AUTHTOKEN", "")
FALLBACK_CLOUDFLARE = True    # start cloudflare if ngrok fails
WARMUP = True                 # send one ping before showing endpoints

# -------------------------
# Live monitoring config
# -------------------------
HEALTH_INTERVAL = 5           # seconds between /health checks
LOG_REFRESH = 1.0             # seconds between log tail refreshes

# launch_backend_live() starts the server, tunnels, and then enters a
# continuous monitoring loop with realtime log streaming and a shutdown button.
# The cell keeps running until you click Shutdown or the server dies.
result = launch_backend_live(
    root_dir=ROOT,
    warmup=WARMUP,
    tunnel_mode=TUNNEL_MODE,
    ngrok_token=NGROK_AUTHTOKEN,
    fallback_cloudflare=FALLBACK_CLOUDFLARE,
    thinking_config=THINKING_CONFIG,
    health_interval=HEALTH_INTERVAL,
    log_refresh=LOG_REFRESH,
    **GENERAL_CONFIG,
    **ADVANCED_CONFIG,
)


In [ ]:
# OPTIONAL — quick status / stop
from gguf_backend.server import stop_server
from gguf_backend.shell import run

# stop_server(ROOT)
run("nvidia-smi --query-gpu=index,name,memory.used,memory.free,utilization.gpu,power.draw --format=csv,noheader,nounits || true", label="gpu status")
run(f"cat {ROOT}/llama_server.pid 2>/dev/null || true", label="server pid")
